In [8]:
from transformers import AutoProcessor, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from time import time

MODEL_ID = "google/gemma-4-E2B-it"

# LOAD MODEL
print("Loading model...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="cuda"
)

print("Model loaded successfully!")

Loading model...


Loading weights: 100%|██████████| 1951/1951 [00:11<00:00, 173.47it/s] 


Model loaded successfully!


In [11]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    }
]

# CONTINUOUS CHAT LOOP

while True:

    # Get user input
    user_input = input("You: ").strip()

    # Exit condition
    if user_input.lower() == "exit":
        print("Assistant: Goodbye!")
        break

    # Skip empty input
    if not user_input:
        continue

    # Add user message to history
    messages.append({
        "role": "user",
        "content": user_input
    })

    start = time()

    try:
        # Process conversation history
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        inputs = processor(
            text=text,
            return_tensors="pt"
        ).to(model.device)

        # Create streamer
        streamer = TextIteratorStreamer(
            processor.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        # Generation arguments
        generation_kwargs = {
            **inputs,
            "max_new_tokens": 2048,
            "streamer": streamer,

            # Sampling parameters
            "do_sample": True,
            "temperature": 1.0,
            "top_p": 0.95,
            "top_k": 64,
        }

        # Run generation in separate thread
        thread = Thread(
            target=model.generate,
            kwargs=generation_kwargs,
        )

        thread.start()

        # Stream assistant response
        response_chunks = []

        print("\nAssistant: ", end="", flush=True)

        for chunk in streamer:
            print(chunk, end="", flush=True)
            response_chunks.append(chunk)

        print()

        # Wait for generation to finish
        thread.join()

        # Combine response chunks
        response = "".join(response_chunks).strip()

        # Save assistant response to history
        messages.append({
            "role": "assistant",
            "content": response
        })

        print(f"Time Taken: {time() - start:.2f} seconds\n")

    except Exception as e:
        print(f"\nError: {e}\n")

        # Remove the failed user message so history stays consistent
        if messages and messages[-1]["role"] == "user":
            messages.pop()


You:  hi



Assistant: Hi there! How can I help you today? 😊
Time Taken: 0.46 seconds



You:  hello



Assistant: Hello! How can I assist you right now? Are you looking to ask a question, need some information, want to brainstorm, or something else?
Time Taken: 0.90 seconds



You:  i need help



Assistant: I'd be happy to help! To give you the best assistance, could you tell me a little more about what you need help with?

For example, are you looking for help with:

* **A specific task** (writing, coding, research)?
* **Information** on a certain topic?
* **Brainstorming** ideas?
* **Troubleshooting** a problem?
* **Creative writing**?
* **Something else entirely?**

The more details you give me, the better I can assist you! 😊
Time Taken: 3.41 seconds



... (2 outputs left out)

You:  what is quantization basic definition



Assistant: Quantization is a fundamental process in various fields, especially in **machine learning, signal processing, and digital communications**.

In its most basic definition, **quantization is the process of mapping a continuous range of real numbers to a finite set of discrete values.**

Think of it like taking something infinitely smooth (continuous) and forcing it to only have specific, countable steps (discrete).

---

### 1. The Core Concept (Analogy)

Imagine you are measuring the temperature of a room.

* **Continuous Representation:** The true temperature could be any value, like $22.3145879...$ degrees Celsius. This is a **continuous** value.
* **Quantization:** You decide you can only measure the temperature to the nearest whole degree.
    * $22.3145879...$ is mapped (or "quantized") to **$22$**.

You have taken an infinite possibility (any real number) and restricted it to a finite set of options (integers, or specific steps).

### 2. Where is Quantization Used?

##

You:  exit


Assistant: Goodbye!
